# CLOAF on a GPU notebook (Colab or Kaggle)

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU, and add the data as a Kaggle dataset instead of Drive (change `DATA` below).

Put the licensed data in Google Drive (never in the repo):

```
MyDrive/cloaf_data/
  smpl/SMPL_NEUTRAL.pkl   (+ SMPL_MALE.pkl, SMPL_FEMALE.pkl)
  amass/<subset>.tar.bz2  (one AMASS sub-dataset, "SMPL+H G")
```

Everything that matters (checkpoints, logs, results) is written to Drive, so a disconnect loses nothing:
reconnect, run the setup cells again and continue.

## Setup (run again after every disconnect)

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists('cloaf_impl'):
    !git clone https://github.com/arhannlp/cloaf_impl.git
%cd /content/cloaf_impl
!git pull -q
!pip install -q torchdiffeq smplx trimesh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/cloaf_data'
!mkdir -p data/body_models data/amass {DATA}/checkpoints {DATA}/results
!cp -rn {DATA}/smpl data/body_models/
!if [ -z "$(ls -A data/amass)" ]; then for f in {DATA}/amass/*.tar.bz2; do tar -xjf "$f" -C data/amass; done; fi
!ls data/body_models/smpl; find data/amass -name '*.npz' | wc -l

## Step 0 – smoke test (fake body, must print SMOKE TEST OK)

In [ ]:
!python tests/smoke_test.py 2>&1 | grep -E 'PASS|FAIL|SKIP|SMOKE'

## Step 1 – Jacobian check on the real SMPL (compare with Fig. 5 / B.7; write the numbers down)

In [ ]:
!python scripts/check_linearity.py --smpl data/body_models/smpl --restarts 3 --out {DATA}/results/linearity.npz

## Step 2 – simple target field, no training (hand-target distance must shrink)

In [ ]:
!python scripts/demo_target_field.py --smpl data/body_models/smpl --out {DATA}/results/target_field.npz

## Step 3 – prepare AMASS and pick test sequences

In [ ]:
!python scripts/prepare_amass.py --amass_root data/amass --out data/amass_smpl_30fps.npz
import glob
SEQS = sorted(glob.glob('data/amass/**/*_poses.npz', recursive=True))
TEST = [SEQS[i] for i in (0, len(SEQS) // 3, 2 * len(SEQS) // 3)]   # 3 different sequences
print(len(SEQS), 'sequences; test on:', *TEST, sep='\n')

## Step 4 – training-free baselines (run while you wait, or before training)

* `point_target`: Supp. C "simple field"; expected to keep the collisions (no non-intersection property).
* `smooth_target`: smooth in space, so touching points move together; our own training-free variant.

In [ ]:
RES = f'{DATA}/results'
for seq in TEST:
    for fld in ['point_target', 'smooth_target']:
        !python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" --field {fld} \
            --frames 90 --corrupt --mode all --out {RES}/$(basename "{seq}" .npz)_{fld}_all.npz 2>&1 | grep -v WARNING

## Step 5 – train the motion field

Re-run this cell to **continue** training (it resumes from Drive). Each run lasts `--minutes`.
The loss must go down; if it does not after ~20 minutes, stop and check.

In [ ]:
!python -m cloaf.train --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz \
    --batch 16 --minutes 120 --resume --out {DATA}/checkpoints/field.pt

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{DATA}/checkpoints/field.pt.log.csv')
plt.semilogy(log.iteration, log.loss); plt.xlabel('iteration'); plt.ylabel('loss (avg of 100)'); plt.grid(); plt.show()
log.tail()

## Step 6 – CLOAF with the trained network, both modes (the project extension)

In [ ]:
for seq in TEST:
    for mode in ['all', 'colliding_only']:
        !python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" \
            --field {DATA}/checkpoints/field.pt --frames 90 --corrupt --mode {mode} \
            --out {RES}/$(basename "{seq}" .npz)_net_{mode}.npz 2>&1 | grep -v WARNING

## Step 7 – summary table (one row per run; also saved as results/summary.csv in Drive)

In [ ]:
import pandas as pd
pd.read_csv(f'{RES}/summary.csv')